In [ ]:
# Langsmith for tracing (Optional)
import getpass
import os

# Prompt to enter the Langsmith API key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [21]:
# -----------------------------
# Chat Model: OpenAI GPT-4o Mini
# -----------------------------

# Prompts the user to enter OPENAI_API_KEY securely.
if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o", model_provider="openai")

In [22]:
# -------------------------
# Embeddings Configuration
# -------------------------

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [23]:
# -----------------------------
# Vector Store Configuration
# -----------------------------

# Create a vector store based on Chroma for semantic similarity search.
from langchain_chroma import Chroma
vector_store = Chroma(embedding_function=embeddings)

In [24]:
# Importing necessary modules

from langchain_core.prompts import ChatPromptTemplate # Provides chat templates but I have used SystemMessage instead of this.
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import MessagesState, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langgraph.graph import END
from langgraph.prebuilt import ToolNode, tools_condition

In [25]:
# ----------------------
# Load the PDF file(s)
# ----------------------

file_path = "data\Structured_RAG.pdf" # Kindly replace with the path to your PDF file
loader = PyPDFLoader(file_path)

In [26]:
# Review page_content for the file
docs = loader.load()

# Inspect the first document to verify its content and metadata.
docs[0]

Document(metadata={'producer': 'Skia/PDF m132', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36', 'creationdate': '2025-02-15T21:55:14+00:00', 'title': 'MyCSV.html', 'moddate': '2025-02-15T21:55:14+00:00', 'source': 'data\\Structured_RAG.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1'}, page_content="Minority Comment ExpertsExplanation\n0 Chakma\n1.In our hill tribes the\nmain ingredient we use\nin our food is 'sidol' in\nenglish it is called\nshrimp paste which is\nmade by several small\nsmashed fish.It has a bit\nstrong smell i agree\nthough.It is typically\nmade in coastal areas\nmainly with seafood.In\nour country its main\nproduction area is\ncoxbazar i guess or other\nsea area.There are other\nsources also i dont have\nthat much idea right now\n2.if i have to talk about\nour food habbit i would\nsay it has come from our\nancestors. 3.in our\nreligion there are no\nrestrictions regarding our\ndietry.

In [27]:
# ---------------------------
# Split the text into chunks
# ---------------------------

text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
all_splits = text_splitter.split_documents(docs)

In [28]:
# -------------
# Index chunks
# -------------

_ = vector_store.add_documents(documents=all_splits)

In [70]:
# --------------------------------
# Define a Retrieval Tool (Method)
# --------------------------------

@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """
    Retrieve information related to a user's query from the vector store.
    
    Args:
        query (str): The search query to execute against the vector store.

    Returns:
        tuple: A tuple containing a serialized representation of the relevant documents 
               and the document objects themselves for further processing.
    """
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n" f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [71]:
# ------------------------------------------------------------------
# Step 1: Generate an AIMessage that always includes a tool-call
# ------------------------------------------------------------------
def query_or_respond(state: MessagesState):
    """Always triggers retrieval regardless of model judgment"""
    # MODIFIED: Explicit system instruction and tool binding
    system_message = SystemMessage(
        "You MUST use the retrieve tool before responding. Never skip retrieval."
    )
    llm_with_tools = llm.bind_tools([retrieve])  # Force tool availability
    
    messages = [system_message] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# ------------------------------------------------
# Step 2: Execute the retrieval via the Tool Node
# ------------------------------------------------
tools = ToolNode([retrieve])


# -----------------------------------------------------
# Step 3: Generate a final response using the retrieved content
# -----------------------------------------------------
def generate(state: MessagesState):
    """Mandatory context usage from retrieval"""
    # MODIFIED: Explicit context extraction
    tool_messages = [msg for msg in state["messages"] if msg.type == "tool"]
    docs_content = "\n\n".join(msg.content for msg in tool_messages[-2:]) 

    system_message_content = (
        """
        As a moderator familiar with religious and ethnic relationships in Bangladesh, assess whether this text represents the country’s broader societal values. Consider if it could be perceived as hate speech, hurtful, or culturally insensitive to members of any community. Provide feedback by highlighting potential issues and suggesting ways to foster respectful and inclusive dialogue. Answer briefly and translate that in the Bengali language before responding."""
        "\n\n"
        f"{docs_content}\n\nUser query: {state['messages'][-1].content}"
    )

    prompt = [SystemMessage(system_message_content)] + state["messages"]
    response = llm.invoke(prompt)
    return {"messages": [response]}

In [72]:
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("query_or_respond", query_or_respond)
graph_builder.add_node("tools", tools)
graph_builder.add_node("generate", generate)

# MODIFIED: Enforced sequence
graph_builder.set_entry_point("query_or_respond")
graph_builder.add_edge("query_or_respond", "tools")
graph_builder.add_edge("tools", "generate")
graph_builder.add_edge("generate", END)

graph = graph_builder.compile()

In [53]:
# -----------------------------
# Example Usage
# -----------------------------
input_message = "ঢাকা শহরে চাকমা পাইলে পাঠানো হবে ঢাকা শহরে কোনো চাকমা যদি পাও তাহলে পিঠিয়ে মারিয়ে ফেলো কোনো চাকমা যদি রাস্তা দেখো তাহলে পিঠিয়ে মারো আমাদের বাঙালি দের কে গুল্লি করতে ছে ওরা"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

ঢাকা শহরে চাকমা পাইলে পাঠানো হবে ঢাকা শহরে কোনো চাকমা যদি পাও তাহলে পিঠিয়ে মারিয়ে ফেলো কোনো চাকমা যদি রাস্তা দেখো তাহলে পিঠিয়ে মারো আমাদের বাঙালি দের কে গুল্লি করতে ছে ওরা
================================== Ai Message ==================================
Tool Calls:
  retrieve (call_PNGr5sVcnX5hu1Zx1oFPP46U)
 Call ID: call_PNGr5sVcnX5hu1Zx1oFPP46U
  Args:
    query: discrimination against the Chakma community in Dhaka
================================= Tool Message =================================
Name: retrieve

Source: {'creationdate': '2025-02-15T21:55:14+00:00', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36', 'moddate': '2025-02-15T21:55:14+00:00', 'page': 6, 'page_label': '7', 'producer': 'Skia/PDF m132', 'source': 'data\\Structured_RAG.pdf', 'title': 'MyCSV.html', 'total_pages': 42}
Content: Chakma people. The speaker is 

In [ ]:
import pandas as pd

# Read the input CSV file
input_df = pd.read_csv('all_prompts.csv')

results = []
count = 0

for prompt in input_df['Prompts']:
    # Retrieve the original data for reference.
    row_data = input_df[input_df['Prompts'] == prompt].to_dict('records')[0]

    input_message = prompt  # Set the current prompt.
    messages = {"human": "", "tool": "", "ai": ""}
    count += 1
    print(f"{count} XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

    # Run the graph with the current prompt
    for step in graph.stream(
        {"messages": [{"role": "user", "content": input_message}]},
        stream_mode="values",
    ):
        current_message = step["messages"][-1]

        # Categorize the message based on its type.
        if current_message.type == "human":
            messages["human"] = current_message.content
        elif current_message.type == "tool":
            messages["tool"] = current_message.content
        elif current_message.type == "ai":
            messages["ai"] = current_message.content

    # Combine the output into a single row.
    result_row = {
        **row_data,
        'Human_Message': messages["human"],
        'Retrieved': messages["tool"],
        'RAG': messages["ai"],
    }
    results.append(result_row)

1 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


In [ ]:
output_df = pd.DataFrame(results)
output_df.to_csv('Output_RAG4.csv',
                index=False,
                escapechar='\\',
                doublequote=True,
                encoding='utf-8-sig'
)
print("Processing complete. Output saved in 'Ouptut_RAG4.csv'.")

Processing complete. Output saved in 'Ouptut_RAG32.csv'.
